# 02 Cleaning

Use this notebook to clean the raw data, document your transformations, and export a processed file to `data/processed/`.

In [2]:
%pip install pandas numpy

253.88s - pydevd: Sending message related to process being replaced timed-out after 5 seconds


  Using cached pandas-3.0.2-cp312-cp312-macosx_11_0_arm64.whl.metadata (79 kB)
  Using cached numpy-2.4.4-cp312-cp312-macosx_14_0_arm64.whl.metadata (6.6 kB)
Using cached pandas-3.0.2-cp312-cp312-macosx_11_0_arm64.whl (9.9 MB)
Using cached numpy-2.4.4-cp312-cp312-macosx_14_0_arm64.whl (5.2 MB)

[notice] A new release of pip is available: 24.2 -> 26.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [ ]:
# imports

from pathlib import Path
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

print(" Libraries imported successfully. Environment initialized for ETL.")

✓ Libraries imported successfully. Environment initialized for ETL.


In [ ]:
# raw data loading

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().resolve().name == 'notebooks' else Path.cwd().resolve()
RAW_PATH = PROJECT_ROOT / 'data' / 'raw' / 'RTA_Dataset.csv'
PROCESSED_PATH = PROJECT_ROOT / 'data' / 'processed' / 'RTA_cleaned.csv'

try:
    df = pd.read_csv(RAW_PATH)
    print(f" Original shape: {df.shape}")
except FileNotFoundError:
    print(f"CRITICAL ERROR: Dataset not found at {RAW_PATH}.")
    raise

✓ Original shape: (12316, 32)


In [ ]:
# column name cleaning
df.columns = df.columns.str.strip().str.replace(' ', '_')
print(" Column names cleaned and standardized.")

✓ Column names cleaned and standardized.


In [ ]:
# cleaning string columns - removing leading/trailing spaces and handling ALL empty values
string_cols = df.select_dtypes(include=['object']).columns
for col in string_cols:
    df[col] = df[col].astype(str).str.strip()

    missing_values = ['na', 'nan', 'NaN', 'NA', 'N/A', 'n/a',
                     'Unknown', 'unknown', 'UNKNOWN',
                     'None', 'none', 'NONE',
                     'null', 'NULL', 'Null',
                     '', ' ', '  ', '   ']

    df[col] = df[col].replace(missing_values, np.nan)
    df[col] = df[col].replace(r'^\s*$', np.nan, regex=True)

print(" String data cleaned - all empty values converted to NaN")
print(f"Missing values after initial clean: {df.isnull().sum().sum()}")

✓ String data cleaned - all empty values converted to NaN
Missing values after initial clean: 43293


In [ ]:
# time conversion to datetime and extracting hour of day
df['Time'] = pd.to_datetime(df['Time'], format='%H:%M:%S', errors='coerce')
df['Hour_of_day'] = df['Time'].dt.hour

print(" Time converted and 'Hour_of_day' extracted")

✓ Time converted and 'Hour_of_day' extracted


In [ ]:
# dropping columns that are not relevant for city planning analysis
columns_to_drop = [
    'Educational_level', 
    'Work_of_casuality', 
    'Fitness_of_casuality', 
    'Vehicle_driver_relation', 
    'Owner_of_vehicle'
]

df = df.drop(columns=columns_to_drop, errors='ignore')

print(f" Scoped dataset for City Planning. Dropped irrelevant columns.")
print(f"Remaining columns: {len(df.columns)}")

✓ Scoped dataset for City Planning. Dropped irrelevant columns.
Remaining columns: 28


In [ ]:
# derieved features for operational insights
# 1. Traffic Patrol Shifts
def map_patrol_shifts(hour):
    if pd.isna(hour):
        return 'Unknown'
    if 6 <= hour < 14:
        return 'Morning Shift (06:00 - 14:00)'
    elif 14 <= hour < 22:
        return 'Evening Shift (14:00 - 22:00)'
    else:
        return 'Night Shift (22:00 - 06:00)'

df['Patrol_Shift'] = df['Hour_of_day'].apply(map_patrol_shifts)

# 2. High-Risk Environment Flag
adverse_weather = ~df['Weather_conditions'].astype(str).str.contains('Normal', case=False, na=False)
adverse_road = ~df['Road_surface_conditions'].astype(str).str.contains('Dry', case=False, na=False)
df['Is_Adverse_Condition'] = np.where(adverse_weather | adverse_road, 'Yes', 'No')

# 3. Weekend Indicator
df['Is_weekend'] = df['Day_of_week'].isin(['Saturday', 'Sunday']).map({True: 'Weekend', False: 'Weekday'})

# 4. Severity Score
severity_map = {'Slight Injury': 1, 'Serious Injury': 2, 'Fatal injury': 3}
df['Severity_score'] = df['Accident_severity'].map(severity_map)

print(" Derived operational features created:")
print("  - Patrol_Shift")
print("  - Is_Adverse_Condition")
print("  - Is_weekend")
print("  - Severity_score")

✓ Derived operational features created:
  - Patrol_Shift
  - Is_Adverse_Condition
  - Is_weekend
  - Severity_score


In [ ]:
# Standardization of Day_of_week and Sex columns
df['Day_of_week'] = df['Day_of_week'].str.capitalize()
df['Sex_of_driver'] = df['Sex_of_driver'].str.capitalize()
df['Sex_of_casualty'] = df['Sex_of_casualty'].str.capitalize()

# Clean Area_accident_occured (remove extra spaces)
if 'Area_accident_occured' in df.columns:
    df['Area_accident_occured'] = df['Area_accident_occured'].str.strip()

print(" Categorical values standardized")

✓ Categorical values standardized


In [ ]:
# Defensive Missing Value Handling & Type Casting

# assigning 'Unknown' to prevent data skew
categorical_cols = df.select_dtypes(include=['object']).columns
for col in categorical_cols:
    df[col] = df[col].fillna('Unknown')
    # Replace empty strings caught late in the pipeline
    df[col] = df[col].replace('', 'Unknown')
    
    # Cast to category for memory optimization and Tableau performance
    df[col] = df[col].astype('category')

# numeric columns - filled with median
numeric_cols = df.select_dtypes(include=[np.number]).columns
for col in numeric_cols:
    if df[col].isnull().sum() > 0:
        df[col] = df[col].fillna(df[col].median())

print(" Missing values handled defensively without artificial skewing.")
print(f"Remaining missing values: {df.isnull().sum().sum()}")


✓ Missing values handled defensively without artificial skewing.
Remaining missing values: 0


In [ ]:
# duplicate removal

duplicates_before = df.duplicated().sum()
df = df.drop_duplicates()
duplicates_removed = duplicates_before - df.duplicated().sum()

print(f" Duplicates removed: {duplicates_removed}")
print(f"Final shape: {df.shape}")

✓ Duplicates removed: 0
Final shape: (12316, 32)


In [ ]:
# Data Quality Validation

print("="*80)
print("DATA QUALITY VALIDATION")
print("="*80)
print(f"\nFinal dataset shape: {df.shape}")
print(f"Total missing values: {df.isnull().sum().sum()}")
print(f"Duplicate rows: {df.duplicated().sum()}")
print("\nData types:")
print(df.dtypes.value_counts())

DATA QUALITY VALIDATION

Final dataset shape: (12316, 32)
Total missing values: 0
Duplicate rows: 0

Data types:
int64             3
category          2
datetime64[us]    1
category          1
category          1
category          1
category          1
category          1
category          1
category          1
category          1
category          1
category          1
category          1
category          1
category          1
category          1
category          1
category          1
category          1
category          1
category          1
category          1
category          1
category          1
int32             1
category          1
category          1
category          1
Name: count, dtype: int64


In [ ]:
# data preview

display(df.head())

,Time,Day_of_week,Age_band_of_driver,Sex_of_driver,Driving_experience,Type_of_vehicle,Service_year_of_vehicle,Defect_of_vehicle,Area_accident_occured,Lanes_or_Medians,...,Age_band_of_casualty,Casualty_severity,Pedestrian_movement,Cause_of_accident,Accident_severity,Hour_of_day,Patrol_Shift,Is_Adverse_Condition,Is_weekend,Severity_score
0,1900-01-01 17:02:00,Monday,18-30,Male,1-2yr,Automobile,Above 10yr,No defect,Residential areas,Unknown,...,Unknown,Unknown,Not a Pedestrian,Moving Backward,Slight Injury,17,Evening Shift (14:00 - 22:00),No,Weekday,1
1,1900-01-01 17:02:00,Monday,31-50,Male,Above 10yr,Public (> 45 seats),5-10yrs,No defect,Office areas,Undivided Two way,...,Unknown,Unknown,Not a Pedestrian,Overtaking,Slight Injury,17,Evening Shift (14:00 - 22:00),No,Weekday,1
2,1900-01-01 17:02:00,Monday,18-30,Male,1-2yr,Lorry (41?100Q),Unknown,No defect,Recreational areas,other,...,31-50,3,Not a Pedestrian,Changing lane to the left,Serious Injury,17,Evening Shift (14:00 - 22:00),No,Weekday,2
3,1900-01-01 01:06:00,Sunday,18-30,Male,5-10yr,Public (> 45 seats),Unknown,No defect,Office areas,other,...,18-30,3,Not a Pedestrian,Changing lane to the right,Slight Injury,1,Night Shift (22:00 - 06:00),No,Weekend,1
4,1900-01-01 01:06:00,Sunday,18-30,Male,2-5yr,Unknown,5-10yrs,No defect,Industrial areas,other,...,Unknown,Unknown,Not a Pedestrian,Overtaking,Slight Injury,1,Night Shift (22:00 - 06:00),No,Weekend,1


In [ ]:
# Save cleaned data

# Ensure processed directory exists
PROCESSED_PATH.parent.mkdir(parents=True, exist_ok=True)

# Save cleaned dataset
df.to_csv(PROCESSED_PATH, index=False)

print("="*80)
print("CLEANING COMPLETE")
print("="*80)
print(f" Cleaned dataset saved to: {PROCESSED_PATH}")
print(f" Final records: {len(df):,}")
print(f" Final features: {len(df.columns)}")
print("\n→ Next step: Proceed to 03_eda.ipynb for exploratory analysis")

CLEANING COMPLETE
✓ Cleaned dataset saved to: /Users/rohankumar/Desktop/DVA-ModelCraft-Retail/data/processed/RTA_cleaned.csv
✓ Final records: 12,316
✓ Final features: 32

→ Next step: Proceed to 03_eda.ipynb for exploratory analysis


In [ ]:
# Generate Cleaning Summary Report

cleaning_summary = f"""
DATA CLEANING SUMMARY
{'='*80}

TRANSFORMATIONS APPLIED:
1. Column names standardized.
2. String data cleaned (whitespace stripped, empty values set to NaN).
3. 'Time' converted safely, extracting 'Hour_of_day'.
4. Irrelevant columns dropped to scope data for City Planning.
5. Derived operational features: Patrol_Shift, Is_Adverse_Condition, Is_weekend, Severity_score.
6. Categorical values explicitly standardized.
7. Defensively imputed missing categoricals with 'Unknown'.
8. Filled missing numeric data with medians.
9. All categories cast correctly for optimized Tableau loading.
10. Duplicate records logged and removed.

DATA QUALITY METRICS:
- Final records: {len(df):,}
- Final features: {len(df.columns)}
- Missing values remaining: {df.isnull().sum().sum()}
- Duplicates remaining: {df.duplicated().sum()}

READY FOR ANALYSIS: Dataset is clean, enriched with operational features, and optimized for Tableau.
"""

summary_path = PROJECT_ROOT / 'data' / 'processed' / 'cleaning_summary.txt'
with open(summary_path, 'w') as f:
    f.write(cleaning_summary)
print(f" Cleaning summary saved to: {summary_path}")

✓ Cleaning summary saved to: /Users/rohankumar/Desktop/DVA-ModelCraft-Retail/data/processed/cleaning_summary.txt
